# Notebook 05 — Restaurant Song Recommender

Takes a restaurant's PCA vector, maps it into Spotify PCA space using a weight matrix, then returns the closest matching songs via cosine similarity.

In [33]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

songs         = pd.read_csv(os.path.join(PROCESSED, 'song_pca.csv'))
restaurants   = pd.read_csv(os.path.join(PROCESSED, 'restaurant_pca.csv'))

print(f"Songs:       {songs.shape}")
print(f"Restaurants: {restaurants.shape}")

Songs:       (566780, 9)
Restaurants: (34808, 15)


---
## Step 1 — Separate identifiers from PC scores

In [34]:
SPOTIFY_PC_COLS = [c for c in songs.columns        if c.startswith('pc')]
YELP_PC_COLS    = [c for c in restaurants.columns  if c.startswith('pc')]

song_vecs       = songs[SPOTIFY_PC_COLS].values          # (1.2M, 6)
restaurant_vecs = restaurants[YELP_PC_COLS].values       # (52K, 13)

print(f"Song vectors:  {song_vecs.shape}")
print(f"Restaurant vectors: {restaurant_vecs.shape}")
print(f"\nSpotify PCs: {SPOTIFY_PC_COLS}")
print(f"Yelp PCs:    {YELP_PC_COLS}")

Song vectors:  (566780, 6)
Restaurant vectors: (34808, 13)

Spotify PCs: ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6']
Yelp PCs:    ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8', 'pc9', 'pc10', 'pc11', 'pc12', 'pc13']


---
## Step 2 — Read the PCA loadings

Before writing the weight matrix we need to know what each PC actually represents.
The loadings tell us which original features drive each component.

In [35]:
pca_spotify = joblib.load(os.path.join(PROCESSED, 'spotify_pca.pkl'))
pca_yelp    = joblib.load(os.path.join(PROCESSED, 'yelp_pca.pkl'))

AUDIO_FEATURES = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'loudness', 'tempo'
]

YELP_FEATURES = [
    'Ambience.romantic', 'Ambience.divey', 'Ambience.classy',
    'Ambience.hipster', 'Ambience.trendy', 'Ambience.upscale', 'Ambience.casual',
    'HasTV', 'HappyHour', 'RestaurantsGoodForGroups',
    'GoodForMeal.breakfast', 'GoodForMeal.brunch',
    'GoodForMeal.latenight', 'GoodForMeal.dinner',
    'RestaurantsTableService', 'NoiseLevel', 'stars'
]

# Trim to however many features survived after dropna / low-variance removal
n_yelp_feats    = pca_yelp.n_features_in_
n_spotify_feats = pca_spotify.n_features_in_
YELP_FEATURES   = YELP_FEATURES[:n_yelp_feats]
AUDIO_FEATURES  = AUDIO_FEATURES[:n_spotify_feats]

spotify_loadings = pd.DataFrame(
    pca_spotify.components_,
    index=SPOTIFY_PC_COLS,
    columns=AUDIO_FEATURES
).round(2)

yelp_loadings = pd.DataFrame(
    pca_yelp.components_,
    index=YELP_PC_COLS,
    columns=YELP_FEATURES
).round(2)

print("=== SPOTIFY loadings ===")
display(spotify_loadings)

print("\n=== YELP loadings ===")
display(yelp_loadings)

=== SPOTIFY loadings ===


,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,loudness,tempo
pc1,0.30,0.48,0.16,-0.43,-0.28,0.13,0.32,0.48,0.19
pc2,0.55,-0.29,0.29,0.33,-0.33,-0.20,0.42,-0.16,-0.27
pc3,-0.19,-0.01,0.58,0.08,-0.17,0.71,-0.18,-0.09,-0.20
pc4,-0.01,-0.14,0.31,0.18,0.03,-0.01,0.09,-0.18,0.90
pc5,0.11,0.12,0.64,-0.26,0.36,-0.50,-0.33,-0.01,-0.13
pc6,0.26,0.08,-0.03,-0.00,0.78,0.35,0.42,-0.14,-0.06



=== YELP loadings ===


,Ambience.romantic,Ambience.divey,Ambience.classy,Ambience.hipster,Ambience.trendy,Ambience.upscale,Ambience.casual,HasTV,HappyHour,RestaurantsGoodForGroups,GoodForMeal.breakfast,GoodForMeal.brunch,GoodForMeal.latenight,GoodForMeal.dinner,RestaurantsTableService,NoiseLevel,stars
pc1,0.13,-0.04,0.32,0.14,0.27,0.15,0.30,0.10,0.37,0.24,0.08,0.18,0.14,0.42,0.42,0.09,0.24
pc2,-0.07,-0.02,-0.08,0.18,0.07,-0.08,0.13,-0.25,-0.22,-0.08,0.63,0.59,-0.03,-0.20,-0.01,-0.08,0.15
pc3,0.40,-0.07,0.34,0.13,0.23,0.36,-0.36,-0.36,-0.06,-0.23,-0.12,-0.07,-0.23,-0.12,-0.02,-0.24,0.24
pc4,0.14,0.20,0.10,0.11,0.20,0.24,-0.35,0.16,0.19,0.00,0.16,0.17,0.24,-0.25,-0.09,0.50,-0.44
pc5,-0.26,0.16,-0.16,0.61,0.41,-0.29,0.04,-0.20,0.06,-0.19,-0.21,-0.21,0.18,-0.01,-0.12,0.13,0.16
pc6,0.05,0.77,0.06,-0.20,-0.23,0.03,-0.06,-0.02,-0.03,-0.23,0.04,0.01,0.38,0.08,0.06,-0.12,0.29
pc7,0.31,-0.29,-0.06,-0.02,-0.12,0.13,0.23,-0.38,-0.23,0.08,-0.01,-0.09,0.63,0.17,-0.27,0.08,-0.10
pc8,0.25,0.35,-0.20,0.30,0.07,0.19,0.10,0.19,-0.18,0.64,-0.01,-0.04,-0.14,-0.07,-0.26,-0.24,0.02
pc9,0.18,0.22,0.01,-0.10,-0.16,-0.10,0.18,-0.39,-0.07,0.11,-0.04,-0.05,-0.44,0.07,-0.03,0.67,0.15
pc10,-0.31,-0.03,-0.21,0.06,-0.02,0.74,0.27,0.19,-0.09,-0.32,-0.01,-0.03,-0.10,0.09,-0.14,0.17,0.15


---
## Step 3 — Weight matrix W

Shape: `(6 Spotify PCs × 13 Yelp PCs)`

Each row is a Spotify PC. Each column is a Yelp PC.
A positive value means: when a restaurant scores high on that Yelp PC, push toward that Spotify PC.
A negative value pulls away from it.

**Update these weights after reading the loadings above.**

In [36]:
N_SPOTIFY = len(SPOTIFY_PC_COLS)   # 6
N_YELP    = len(YELP_PC_COLS)      # 13

#                  yPC1   yPC2   yPC3   yPC4   yPC5   yPC6   yPC7   yPC8   yPC9  yPC10  yPC11  yPC12  yPC13
# W = np.array([
#     # sPC1          casual/energetic axis
#     [ 0.8,  -0.3,   0.2,   0.1,   0.0,   0.0,   0.0,   0.0,   0.1,   0.0,   0.0,   0.0,   0.0],
#     # sPC2
#     [-0.5,   0.6,  -0.2,   0.2,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
#     # sPC3
#     [ 0.2,   0.1,   0.7,  -0.3,   0.0,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
#     # sPC4
#     [ 0.1,  -0.2,   0.1,   0.6,  -0.2,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
#     # sPC5
#     [ 0.0,   0.1,  -0.1,   0.1,   0.5,   0.2,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
#     # sPC6
#     [ 0.0,   0.0,   0.1,  -0.1,   0.2,   0.4,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
# ], dtype=float)   # shape: (6, 13)


W = np.array([
    [  0.00,   0.00,  -1.00,   1.00,   0.00,   1.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  1.00,   1.00,   0.00,   0.00,   0.80,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.50,   0.00,  -0.50,   0.80,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.00,  -0.50,   1.00,   0.00,   0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.00,   0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  1.00,   0.00,   1.00,  -1.00,   0.00,  -0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
], dtype=float)

assert W.shape == (N_SPOTIFY, N_YELP), f"W must be ({N_SPOTIFY}, {N_YELP}), got {W.shape}"

W_df = pd.DataFrame(W, index=SPOTIFY_PC_COLS, columns=YELP_PC_COLS)
print("Weight matrix W  (rows=Spotify PCs, columns=Yelp PCs):")
display(W_df)

Weight matrix W  (rows=Spotify PCs, columns=Yelp PCs):


,pc1,pc2,pc3,pc4,pc5,pc6,pc7,pc8,pc9,pc10,pc11,pc12,pc13
pc1,0.0,0.0,-1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc2,1.0,1.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc3,0.0,0.5,0.0,-0.5,0.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc4,0.0,0.0,-0.5,1.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc5,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc6,1.0,0.0,1.0,-1.0,0.0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Step 4 — Trial run

In [26]:
idx = 0
restaurant_row = restaurants.iloc[idx]

# ── STEP 1: raw Yelp features ─────────────────────────────────────────────
yelp_raw  = pd.read_csv(os.path.join(PROCESSED, 'yelp_ml_ready.csv'))
yelp_match = yelp_raw[yelp_raw['business_id'] == restaurant_row['business_id']]
print(f"Restaurant: {restaurant_row['name']}")
print("\n── STEP 1: Raw Yelp features ──")
display(yelp_match[YELP_FEATURES].T.rename(columns={yelp_match.index[0]: 'value'}))

# ── STEP 2: Yelp PC scores ────────────────────────────────────────────────
y = restaurant_vecs[idx]   # already computed by PCA sandbox
print("\n── STEP 2: Yelp PC scores (what PCA compressed the features into) ──")
display(pd.Series(y.round(3), index=YELP_PC_COLS).to_frame('score'))

# ── STEP 3: project into Spotify space via W ──────────────────────────────
s_hat = W @ y
print("\n── STEP 3: Target Spotify PC scores  (s_hat = W @ y) ──")
display(pd.Series(s_hat.round(3), index=SPOTIFY_PC_COLS).to_frame('target'))

# ── STEP 4: what does s_hat mean in audio terms? ─────────────────────────
print("\n── STEP 4: What that target means in audio terms ──")
audio_target = pd.Series(
    pca_spotify.inverse_transform(s_hat.reshape(1, -1))[0],
    index=AUDIO_FEATURES
).round(3)
display(audio_target.to_frame('audio value (standardised)'))

# ── STEP 5: cosine similarity → top 10 songs ─────────────────────────────
scores  = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]
top_idx = np.argsort(scores)[::-1][:10]
results = songs.iloc[top_idx][['name', 'artists']].copy()
results.insert(0, 'score', scores[top_idx].round(3))
print("\n── STEP 5: Top 10 recommended songs ──")
display(results)

Restaurant: Tsevi's Pub And Grill

── STEP 1: Raw Yelp features ──


,value
Ambience.romantic,0.0
Ambience.divey,0.0
Ambience.classy,0.0
Ambience.hipster,0.0
Ambience.trendy,0.0
Ambience.upscale,0.0
Ambience.casual,0.0
HasTV,1.0
HappyHour,0.0
RestaurantsGoodForGroups,1.0



── STEP 2: Yelp PC scores (what PCA compressed the features into) ──


,score
pc1,-1.454
pc2,-0.521
pc3,-0.196
pc4,0.685
pc5,-0.291
pc6,-0.520
pc7,-0.250
pc8,0.475
pc9,-0.283
pc10,-0.236



── STEP 3: Target Spotify PC scores  (s_hat = W @ y) ──


,target
pc1,0.361
pc2,-2.208
pc3,-0.836
pc4,0.523
pc5,-0.098
pc6,-2.075



── STEP 4: What that target means in audio terms ──


,audio value (standardised)
danceability,-1.341
energy,0.611
speechiness,-0.450
acousticness,-0.965
instrumentalness,-0.575
liveness,-1.265
valence,-1.465
loudness,0.706
tempo,1.554



── STEP 5: Top 10 recommended songs ──


,score,name,artists
211853,0.997,Always And Forever,['Randy Muller Boom Chang Bang']
250220,0.996,No Calming Ride,['Roebeck']
652272,0.994,Awake,['Nick Boyd']
150322,0.994,Wet Cigarette,['Metropolitan']
81601,0.994,Delusion,['LoveLikeFire']
976941,0.993,Hallucination,['Martin George Selwood']
648908,0.992,Katzenjammer kids,['Katzenjammer Kabarett']
935465,0.992,The Slow Descent,['Haive Music']
82416,0.991,Butterflies Are Blue,['Magneta Lane']
834154,0.990,The Loser,['Gabe Wright']


In [27]:
import os

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"
yelp_raw  = pd.read_csv(os.path.join(PROCESSED, 'yelp_ml_ready.csv'))

# pick a restaurant to test
idx = 0
restaurant_row = restaurants.iloc[idx]
print(f"Restaurant: {restaurant_row['name']}\n")

# show its raw feature ratings
yelp_match   = yelp_raw[yelp_raw['business_id'] == restaurant_row['business_id']]
feature_vals = yelp_match[YELP_FEATURES].T
feature_vals.columns = ['value']
print("Feature profile:")
display(feature_vals)

# step 1 — grab its Yelp PC vector
y = restaurant_vecs[idx]                                          # shape (13,)

# step 2 — project into Spotify PC space
s_hat = W @ y                                                     # shape (6,)

# step 3 — cosine similarity against every song
scores = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]   # shape (1.2M,)

# step 4 — top 10 results
top_idx = np.argsort(scores)[::-1][:10]
results  = songs.iloc[top_idx][['name', 'artists']].copy()
results.insert(0, 'score', scores[top_idx].round(3))

print("\nTop 10 recommended songs:")
display(results)

Restaurant: Tsevi's Pub And Grill

Feature profile:


,value
Ambience.romantic,0.0
Ambience.divey,0.0
Ambience.classy,0.0
Ambience.hipster,0.0
Ambience.trendy,0.0
Ambience.upscale,0.0
Ambience.casual,0.0
HasTV,1.0
HappyHour,0.0
RestaurantsGoodForGroups,1.0



Top 10 recommended songs:


,score,name,artists
211853,0.997,Always And Forever,['Randy Muller Boom Chang Bang']
250220,0.996,No Calming Ride,['Roebeck']
652272,0.994,Awake,['Nick Boyd']
150322,0.994,Wet Cigarette,['Metropolitan']
81601,0.994,Delusion,['LoveLikeFire']
976941,0.993,Hallucination,['Martin George Selwood']
648908,0.992,Katzenjammer kids,['Katzenjammer Kabarett']
935465,0.992,The Slow Descent,['Haive Music']
82416,0.991,Butterflies Are Blue,['Magneta Lane']
834154,0.990,The Loser,['Gabe Wright']


---
## Step 5 — Sanity-check W across restaurant vibes

Test on restaurants with clearly different profiles. If W is working, an upscale romantic spot and a late-night dive should get noticeably different songs.

In [28]:
TEST_RESTAURANTS = [
    (2260,  "Upscale / Romantic — Laurel Oak Southern Brasserie"),
    (428,   "Upscale — Persian Room Fine Dining"),
    (2668,  "Divey — Chicago Paulies"),
    (585,   "Casual — Pia Urban Cafe & Market"),
    (7127,  "Hipster — Momma Donna"),
    (1872,  "Brunch — Cafe Soleil"),
    (569,   "Late-night — Dots Diner"),
]

for pca_idx, label in TEST_RESTAURANTS:
    y     = restaurant_vecs[pca_idx]
    s_hat = W @ y
    scores  = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]
    top_idx = np.argsort(scores)[::-1][:5]

    top_songs = songs.iloc[top_idx][['name', 'artists']].copy()
    top_songs.insert(0, 'score', scores[top_idx].round(3))
    top_songs = top_songs.reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    display(top_songs)


  Upscale / Romantic — Laurel Oak Southern Brasserie


,score,name,artists
0,0.998,Havana - Instrumental,['Lullaby Players']
1,0.998,Moonlight,['Daniel Roth']
2,0.996,Whistling Rufus,['DeWayne Wear']
3,0.996,Maloya,"['Hans Ludemann', 'Trio Ivoire']"
4,0.995,Whistle Rufus,['DeWayne Wear']



  Upscale — Persian Room Fine Dining


,score,name,artists
0,0.996,Will the Circle Be Unbroken,['Hosanna Plath']
1,0.995,"Aria With 30 Variations Bmv 988, Variation 16",['Gustav Leonhardt']
2,0.994,Whistle Rufus,['DeWayne Wear']
3,0.992,Magic City Blue,['Sun Ra']
4,0.992,Time Is on My Side,['Wilson Pickett']



  Divey — Chicago Paulies


...

,score,name,artists
0,0.988,Intro,['Hex One']
1,0.971,Oliver!,"['Willoughby Goddard', 'Hope Jackman', 'Bruce ..."
2,0.970,"Don Giovanni, K. 527: Act I Scene 16: Recitati...","['Wolfgang Amadeus Mozart', 'Bo Skovhus', 'Jan..."
3,0.965,Trouble on the Water... - Live,['Joe Stilgoe']
4,0.965,Like the Sunshine,['James R. Gray']



  Brunch — Cafe Soleil


,score,name,artists
0,0.975,"Chinatown, My Chinatown",['Louis Armstrong']
1,0.965,Jim Along Josie,['Enzo Garcia']
2,0.963,Natural Facts,['Cornell Campbell']
3,0.956,Tender,['Rockabye Baby!']
4,0.953,Smoked Out (Green Blaze Subliminal Sounds),['Madlib']



  Late-night — Dots Diner


,score,name,artists
0,0.980,On Point,['Ali As']
1,0.979,Cold World,['Street Military']
2,0.972,The Duck's Yas Yas Yas,['Jim Byrnes']
3,0.971,Hands In The Sand,['Manifest Nexto Me']
4,0.970,Mortal Monogamy,"[""Mr. Muthafuckin' eXquire""]"


---
## Step 6 — Boss-Level Archetype Evaluation

PCA extremes are the mathematically purest examples of each dimension.

**The logic:**
- The restaurant with the highest score on yPC2 *is* the most extreme brunch spot in the dataset — by definition.
- The song with the highest score on sPC1 *is* the loudest/most energetic song — by definition.
- If W is working, extreme restaurant on yPC_k → s_hat should strongly activate the Spotify PC that yPC_k is supposed to map to → recommended songs should look like the extreme song on that Spotify PC.

No labels needed. The math tells you whether the connections hold.

In [29]:
# ── Human-readable labels drawn from the loadings printed in Step 2 ───────
YELP_PC_LABELS = {
    'pc1':  'sit-down dinner (table service + casual)',
    'pc2':  'brunch / breakfast',
    'pc3':  'fine dining (upscale + classy + romantic)',
    'pc4':  'loud / mixed (high noise, lower stars)',
    'pc5':  'hipster / trendy',
    'pc6':  'dive bar',
    'pc7':  'late-night',
    'pc8':  'good for groups',
}

SPOTIFY_PC_LABELS = {
    'pc1': 'energetic + loud + danceable',
    'pc2': 'happy + danceable (acoustic-leaning)',
    'pc3': 'live performance + speechy',
    'pc4': 'fast tempo',
    'pc5': 'spoken word / pure instrumental',
    'pc6': 'live instrumental / jazz-like',
}

# ── Boss restaurants: max score per Yelp PC ────────────────────────────────
print("=== BOSS-LEVEL RESTAURANTS (max score per Yelp PC) ===\n")
boss_restaurants = {}
for pc in list(YELP_PC_LABELS.keys()):
    idx = restaurants[pc].idxmax()
    name = restaurants.loc[idx, 'name']
    score = restaurants.loc[idx, pc]
    boss_restaurants[pc] = idx
    label = YELP_PC_LABELS[pc]
    print(f"  {pc} ({label})\n    → {name}  (score: {score:.3f})\n")

# ── Boss songs: max score per Spotify PC ──────────────────────────────────
print("\n=== BOSS-LEVEL SONGS (max score per Spotify PC) ===\n")
boss_songs = {}
for pc in list(SPOTIFY_PC_LABELS.keys()):
    idx = songs[pc].idxmax()
    name  = songs.loc[idx, 'name']
    artist = songs.loc[idx, 'artists']
    score  = songs.loc[idx, pc]
    boss_songs[pc] = idx
    label = SPOTIFY_PC_LABELS[pc]
    print(f"  {pc} ({label})\n    → \"{name}\" by {artist}  (score: {score:.3f})\n")

=== BOSS-LEVEL RESTAURANTS (max score per Yelp PC) ===

  pc1 (sit-down dinner (table service + casual))
    → The BAO  (score: 8.436)

  pc2 (brunch / breakfast)
    → Living Room Coffee & Kitchen  (score: 6.836)

  pc3 (fine dining (upscale + classy + romantic))
    → 1799 Kitchen and Cocktails  (score: 10.294)

  pc4 (loud / mixed (high noise, lower stars))
    → Effervescence  (score: 6.751)

  pc5 (hipster / trendy)
    → Bok Bar  (score: 7.208)

  pc6 (dive bar)
    → Verti Marte  (score: 7.382)

  pc7 (late-night)
    → Barcelona Wine Bar  (score: 6.590)

  pc8 (good for groups)
    → Strange Bird  (score: 5.157)


=== BOSS-LEVEL SONGS (max score per Spotify PC) ===

  pc1 (energetic + loud + danceable)
    → "Put Cha D. In Her Mouth - Explicit Album Version" by ['Three 6 Mafia']  (score: 3.786)

  pc2 (happy + danceable (acoustic-leaning))
    → "Commentary on First Suite (2)" by ['Mark Jenkins']  (score: 5.921)

  pc3 (live performance + speechy)
    → "Text Message War" by ['

In [30]:
# ── Evaluation: does each boss restaurant activate the right Spotify PC? ──
print("=== BOSS-LEVEL MATCHUP EVALUATION ===")
print("For each extreme restaurant, check which Spotify PC dominates s_hat")
print("and whether the top songs resemble the boss song for that PC.\n")

for yelp_pc, rest_idx in boss_restaurants.items():
    rest_name = restaurants.loc[rest_idx, 'name']
    y     = restaurant_vecs[rest_idx]
    s_hat = W @ y

    # which Spotify PC is most activated?
    dominant_spc = SPOTIFY_PC_COLS[np.argmax(np.abs(s_hat))]
    dominant_val = s_hat[np.argmax(np.abs(s_hat))]

    # top 3 recommended songs
    scores  = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]
    top_idx = np.argsort(scores)[::-1][:3]

    # boss song for the dominant Spotify PC
    boss_idx    = boss_songs[dominant_spc]
    boss_name   = songs.loc[boss_idx, 'name']
    boss_artist = songs.loc[boss_idx, 'artists']
    boss_rank   = int(np.where(np.argsort(scores)[::-1] == boss_idx)[0][0]) + 1 \
                  if boss_idx in np.argsort(scores)[::-1][:500] else '>500'

    print(f"{'─'*65}")
    print(f"  Restaurant ({yelp_pc} — {YELP_PC_LABELS[yelp_pc]})")
    print(f"  → {rest_name}")
    print(f"  Dominant Spotify PC: {dominant_spc} ({SPOTIFY_PC_LABELS[dominant_spc]})  val={dominant_val:.3f}")
    print(f"  s_hat: { {pc: round(v,2) for pc, v in zip(SPOTIFY_PC_COLS, s_hat)} }")
    print(f"  Boss song for {dominant_spc}: \"{boss_name}\" by {boss_artist}  [ranked #{boss_rank} in recs]")
    print(f"  Top 3 recommended songs:")
    for rank, i in enumerate(top_idx, 1):
        print(f"    {rank}. \"{songs.loc[i,'name']}\" by {songs.loc[i,'artists']}  (cos={scores[i]:.3f})")
    print()

=== BOSS-LEVEL MATCHUP EVALUATION ===
For each extreme restaurant, check which Spotify PC dominates s_hat
and whether the top songs resemble the boss song for that PC.

─────────────────────────────────────────────────────────────────
  Restaurant (pc1 — sit-down dinner (table service + casual))
  → The BAO
  Dominant Spotify PC: pc6 (live instrumental / jazz-like)  val=11.110
  s_hat: {'pc1': np.float64(-3.33), 'pc2': np.float64(10.67), 'pc3': np.float64(-2.2), 'pc4': np.float64(1.14), 'pc5': np.float64(3.82), 'pc6': np.float64(11.11)}
  Boss song for pc6: "Opry Theme" by ['Harold Morrison']  [ranked #>500 in recs]
  Top 3 recommended songs:
    1. "I'm Somebody's Somebody Now" by ['Annette Hanshaw']  (cos=0.993)
    2. "Mtima" by ['Madlib']  (cos=0.992)
    3. "So the Bluebirds and the Blackbirds Got Together" by ['Ray Noble']  (cos=0.991)

─────────────────────────────────────────────────────────────────
  Restaurant (pc2 — brunch / breakfast)
  → Living Room Coffee & Kitchen
  Domi

In [37]:
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler

print("=== BOSS-LEVEL MATCHUP EVALUATION (FIXED MATH) ===")
print("Using Normalized Vectors, StandardScaler, and Euclidean Distance.\n")

# 1. Bypass Popularity Filter (since the column doesn't exist)
# We will use the entire dataset. The Euclidean math will fix the extreme outliers!
recognizable_songs = songs.copy()
song_vecs_filtered = song_vecs.copy()

# 2. Standardize the Spotify Space (CRITICAL for Euclidean distance)
scaler = StandardScaler()
song_vecs_scaled = scaler.fit_transform(song_vecs_filtered)

# ... existing code ...


for yelp_pc, rest_idx in boss_restaurants.items():
    rest_name = restaurants.loc[rest_idx, 'name']
    y = restaurant_vecs[rest_idx]
    
    # FIX A: Normalize the Yelp vector so s_hat doesn't explode
    y_scaled = y / (np.linalg.norm(y) + 1e-9)
    
    # Predict target vector
    s_hat = W @ y_scaled
    
    # FIX B: Scale the target prediction into the standardized Spotify space
    s_hat_scaled = scaler.transform(s_hat.reshape(1, -1))

    dominant_spc = SPOTIFY_PC_COLS[np.argmax(np.abs(s_hat))]
    dominant_val = s_hat[np.argmax(np.abs(s_hat))]

    # FIX C: Use Euclidean distance (lower is better)
    distances = pairwise_distances(s_hat_scaled, song_vecs_scaled, metric='euclidean')[0]
    
    # Get the top 3 closest songs (smallest distance)
    top_idx = np.argsort(distances)[:3]

    print(f"{'─'*65}")
    print(f"  Restaurant ({yelp_pc} — {YELP_PC_LABELS[yelp_pc]})")
    print(f"  → {rest_name}")
    print(f"  Dominant Spotify PC: {dominant_spc} ({SPOTIFY_PC_LABELS[dominant_spc]})  val={dominant_val:.3f}")
    print(f"  s_hat (normalized): { {pc: round(v,2) for pc, v in zip(SPOTIFY_PC_COLS, s_hat)} }")
    print(f"  Top 3 recommended songs:")
    
    for rank, i in enumerate(top_idx, 1):
        # Using recognizable_songs dataframe here
        song_name = recognizable_songs.loc[i, 'name']
        song_artist = recognizable_songs.loc[i, 'artists']
        dist_score = distances[i]
        print(f"    {rank}. \"{song_name}\" by {song_artist}  (dist={dist_score:.3f})")
    print()

=== BOSS-LEVEL MATCHUP EVALUATION (FIXED MATH) ===
Using Normalized Vectors, StandardScaler, and Euclidean Distance.

─────────────────────────────────────────────────────────────────
  Restaurant (pc1 — sit-down dinner (table service + casual))
  → The BAO
  Dominant Spotify PC: pc6 (live instrumental / jazz-like)  val=0.718
  s_hat (normalized): {'pc1': np.float64(-0.22), 'pc2': np.float64(0.69), 'pc3': np.float64(-0.14), 'pc4': np.float64(0.07), 'pc5': np.float64(0.25), 'pc6': np.float64(0.72)}
  Top 3 recommended songs:
    1. "Juan Colorado" by ['Alborada All-Stars']  (dist=0.490)
    2. "Balla Trazan Apansson" by ['Barnens favoriter']  (dist=0.498)
    3. "Instigations" by ["Ray Anderson's Organic Quartet", 'Ray Anderson', 'Tommy Campbell', 'Steve Salerno', 'Gary Versace']  (dist=0.517)

─────────────────────────────────────────────────────────────────
  Restaurant (pc2 — brunch / breakfast)
  → Living Room Coffee & Kitchen
  Dominant Spotify PC: pc2 (happy + danceable (acoustic-

In [32]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. Define the extreme archetypes based on your Cheat Sheet
# We will create a synthetic dataset of 6 extreme restaurants (X) 
# and their perfect matching target Spotify vectors (Y).

# Create an empty 6x13 array for our synthetic restaurants
X_train = np.zeros((6, 13))

# Fill in the extremes (Assuming a normalized scale of 1.0)
X_train[0, 0] = 1.0  # yPC1: Standard Sit-Down
X_train[1, 1] = 1.0  # yPC2: Brunch
X_train[2, 2] = 1.0  # yPC3: Fine Dining
X_train[3, 3] = 1.0  # yPC4: Late-Night Dive
X_train[4, 4] = 1.0  # yPC5: Hipster
X_train[5, 5] = 1.0  # yPC6: Divey

# 2. Define the perfect Spotify target vectors (Y) for those restaurants
# Shape: (6, 6) -> 6 restaurants, 6 target Spotify PCs
Y_train = np.zeros((6, 6))

# Define the targets based on logical personas:
# Target for yPC1 (Sit-Down): High Chill Groove (sPC2), High Jazz (sPC6)
Y_train[0] = [0.0, 1.0, 0.0, 0.0, 0.0, 1.0] 

# Target for yPC2 (Brunch): Perfect Chill Groove (sPC2), Some Acoustic (sPC3)
Y_train[1] = [0.0, 1.0, 0.5, 0.0, 0.0, 0.0] 

# Target for yPC3 (Fine Dining): Very High Jazz/Inst (sPC6), NEGATIVE Club Banger (sPC1)
Y_train[2] = [-1.0, 0.0, 0.0, -0.5, 0.5, 1.0]

# Target for yPC4 (Loud Dive): Very High Club Banger (sPC1), Fast (sPC4), NEGATIVE Jazz (sPC6)
Y_train[3] = [1.0, 0.0, -0.5, 1.0, 0.0, -1.0]

# Target for yPC5 (Hipster): Chill Groove (sPC2), Acoustic (sPC3)
Y_train[4] = [0.0, 0.8, 0.8, 0.0, 0.0, 0.0]

# Target for yPC6 (Divey): High Club Banger (sPC1)
Y_train[5] = [1.0, 0.0, 0.0, 0.5, 0.0, -0.5]


# 3. Train the Model to learn the optimal W matrix
# fit_intercept=False is required so we don't add invisible constants to the math
model = LinearRegression(fit_intercept=False)
model.fit(X_train, Y_train)

# 4. Extract the newly optimized W matrix
# scikit-learn's coef_ naturally outputs (n_targets, n_features), which is exactly (6, 13)!
W_optimized = model.coef_

print("=== NEW OPTIMIZED W MATRIX ===")
print("Copy this array into your main application:\n")
print("W = np.array([")
for row in W_optimized:
    row_str = ", ".join([f"{val:6.2f}" for val in row])
    print(f"    [{row_str}],")
print("], dtype=float)")

=== NEW OPTIMIZED W MATRIX ===
Copy this array into your main application:

W = np.array([
    [  0.00,   0.00,  -1.00,   1.00,   0.00,   1.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  1.00,   1.00,   0.00,   0.00,   0.80,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.50,   0.00,  -0.50,   0.80,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.00,  -0.50,   1.00,   0.00,   0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  0.00,   0.00,   0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
    [  1.00,   0.00,   1.00,  -1.00,   0.00,  -0.50,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00,   0.00],
], dtype=float)
